In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error


In [2]:
df = pd.read_csv("uber.csv")

In [3]:
df.dropna(inplace=True)

In [4]:
df.drop(
    columns = [
        "Unnamed: 0","key"
    ], errors="ignore",inplace=True
)

In [5]:
df.head()

,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


In [6]:
df = df[
    (df["fare_amount"] > 0 ) & (df["passenger_count"] > 0) & (df["passenger_count"] <=6) 
]

In [7]:
df = df[
    df["pickup_latitude"].between(40,42) &
    df["pickup_longitude"].between(-75,-72) &
    df["dropoff_latitude"].between(40,42) &
    df["dropoff_longitude"].between(-75,-72)
]

In [8]:
Q1 = df["fare_amount"].quantile(0.25)
Q3 = df["fare_amount"].quantile(0.75)
IQR = Q3 - Q1

lower = max(2.5, Q1-1.5*IQR)
upper = Q3 + 1.5*IQR

df = df[df["fare_amount"].between(lower,upper)]

In [9]:
print(
    df.corr(numeric_only = True)["fare_amount"]
)

fare_amount          1.000000
pickup_longitude     0.001234
pickup_latitude     -0.075734
dropoff_longitude    0.073949
dropoff_latitude    -0.086855
passenger_count      0.012419
Name: fare_amount, dtype: float64


In [10]:
df.head()

,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


In [11]:
features = [
    "pickup_longitude", "pickup_latitude","dropoff_longitude","dropoff_latitude","passenger_count" 
]

In [12]:
X = df[features]
y = df["fare_amount"]

X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size = 0.2, random_state = 42
)


In [13]:
print("DataFrame:", "df" in globals())
print("X_train:", "X_train" in globals())
print("y_train:", "y_train" in globals())
print("lr:", "lr" in globals())
print("rf:", "rf" in globals())

DataFrame: True
X_train: True
y_train: True
lr: False
rf: False


In [14]:
# Create and train Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)

print("Linear Regression trained successfully!")

# Create and train Random Forest
rf = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

print("Random Forest trained successfully!")

Linear Regression trained successfully!
Random Forest trained successfully!


In [15]:
for name, model in [
    ("Linear Regression", lr),
    ("Random Forest", rf)
]:
    pred = model.predict(X_test)

    r2 = r2_score(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))

    print(name, "R²:", round(r2, 3), "RMSE:", round(rmse, 2))

Linear Regression R²: 0.03 RMSE: 4.08
Random Forest R²: 0.72 RMSE: 2.19
